# XCoder-0.8B - Fine-tune shard-by-shard di Colab T4

Setiap shard `datasets/shred/<stack>/shred-NNNNN.jsonl` diproses satu demi satu:
**list -> audit -> dedupe -> filter -> train 1 epoch -> checkpoint -> shard seterusnya**.

**Panduan**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Jalankan sel dari atas ke bawah.
3. `BASE` mesti model **safetensors** (`Qwen/Qwen3.5-0.8B`). Fail `.gguf`
   (cth. `xcoder-gen1-0.8b-q4_k_m.gguf`) ialah inferens sahaja dan TIDAK boleh dilatih.
4. Output dalam `OUT_DIR`: `state.json`, `clean/`, `ckpt/`, `merged/`, `gguf/`.
5. `STREAM = True` -> tanpa clone: script diambil dari GitHub, setiap shard dimuat,
   dilatih, kemudian dipadam (`--delete-after`); `clean/` + `ckpt/` kekal di Drive.
6. Sesi putus? Buka semula, jalankan semula sel **Train** - shard siap dilangkau dan
   training sambung dari adapter terakhir. Jangan buka dua sesi ke `OUT_DIR` yang sama.

7. **Akaun Google:** pastikan akaun yang dipilih semasa `drive.mount` SAMA
   dengan akaun yang menyimpan `DriveXcoder`. Jika tidak, checkpoint ditulis
   ke Drive akaun lain dan nampak "hilang". Sel **DRIVE CHECK** di bawah
   mengesan ini.

**Beberapa akaun Google:** kongsi `DriveXcoder/xcoder`, `Add shortcut to Drive` di akaun B,
kemudian guna `OUT_DIR` yang sama.

In [ ]:
# ===== CONFIG (edit di sini sahaja) =====
BASE = 'Qwen/Qwen3.5-0.8B'                                  # safetensors, bukan GGUF
OUT_DIR = '/content/drive/MyDrive/DriveXcoder/xcoder/v2'  # checkpoint + export
ARGS = ''                    # cth: '--thinking' atau '--cap 4000' atau '--no-export'
                             # pengetahuan baru: '--stacks new --epochs 2 --lr 5e-5 --replay 0.2 --eval-ratio 0.01'
STREAM = False               # True = tanpa clone: muat 1 shard -> train -> padam -> next
WORK = '/content/train-my-ai'
REPO = 'https://github.com/qubexs/train-my-ai.git'
# ========================================

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# DRIVE CHECK: pastikan akaun Drive betul + OUT_DIR boleh ditulis.
# Jika akaun salah: Runtime -> Disconnect, kemudian mount semula dan pilih
# akaun yang menyimpan DriveXcoder (guna tetingkap Incognito jika perlu).
import glob, os, shutil

OUT_DIR = globals().get('OUT_DIR', '/content/drive/MyDrive/DriveXcoder/xcoder/v2')
print('OUT_DIR:', OUT_DIR)
print('wujud   :', os.path.isdir(OUT_DIR))
if os.path.isdir(OUT_DIR):
    print('isi     :', sorted(os.listdir(OUT_DIR))[:8])
    try:
        print('bebas   : %.1f GB' % (shutil.disk_usage(OUT_DIR).free / 1024 ** 3))
    except OSError as e:
        print('bebas   : gagal', e)
    print('boleh tulis:', os.access(OUT_DIR, os.W_OK))

hits = glob.glob('/content/drive/MyDrive/DriveXcoder/**/state.json',
                 recursive=True)
print('state.json dijumpai:', hits[:5] if hits else 'TIADA')
ck = glob.glob('/content/drive/MyDrive/DriveXcoder/**/ckpt/shard-*',
               recursive=True)
print('ckpt shard:', len(ck), ck[-2:] if ck else '')

if os.path.isdir(OUT_DIR) and not os.listdir(OUT_DIR):
    print()
    print('AMARAN: OUT_DIR kosong. Jika ini bukan larian pertama, akaun Drive')
    print('mungkin SALAH - cari state.json di akaun lain (Drive search: state.json),')
    print('atau mulakan larian baharu di akaun ini (state.json akan dicipta).')

In [ ]:
# Setup: STREAM=False -> clone repo; STREAM=True -> ambil train_shreds.py sahaja
import os

# Fallback jika sel CONFIG belum dijalankan (cth. selepas Restart session)
STREAM = globals().get('STREAM', False)
REPO = globals().get('REPO', 'https://github.com/qubexs/train-my-ai.git')
WORK = globals().get('WORK', '/content' if STREAM else '/content/train-my-ai')

if STREAM:
    WORK = '/content'
    SCRIPT = '/content/train_shreds.py'
    url = 'https://raw.githubusercontent.com/qubexs/train-my-ai/master/finetune/train_shreds.py'
    !wget -q {url} -O {SCRIPT}
    print('stream mode (1 shard -> train -> delete):', SCRIPT)
else:
    SCRIPT = 'finetune/train_shreds.py'
    if not os.path.isdir(f'{WORK}/.git'):
        !git clone --depth 1 {REPO} {WORK}
    else:
        !git -C {WORK} fetch --depth 1 origin master
        !git -C {WORK} reset --hard origin/master
    print('shard folders:', len(os.listdir(f'{WORK}/datasets/shred')))
%cd {WORK}
print('SCRIPT =', SCRIPT, '| WORK =', WORK)

In [ ]:
# Senarai shard yang akan dilatih (62 fail)
BASE = globals().get('BASE', 'Qwen/Qwen3.5-0.8B')
STREAM = globals().get('STREAM', False)
WORK = globals().get('WORK', '/content' if STREAM else '/content/train-my-ai')
SCRIPT = globals().get('SCRIPT', '/content/train_shreds.py' if STREAM else 'finetune/train_shreds.py')
%cd {WORK}
!python {SCRIPT} --model {BASE} --list-only

In [ ]:
# Pasang dependency Unsloth Qwen3.5 (pins tepat). Skip jika semua sudah ada.
import importlib.metadata as md, shutil
from packaging.specifiers import SpecifierSet

PINS = {
    'torch': '==2.8.0', 'triton': '>=3.3.0', 'torchvision': None,
    'torchaudio': '==2.8.0', 'bitsandbytes': None,
    'xformers': '==0.0.32.post2', 'torchcodec': '==0.7.0',
    'unsloth': None, 'unsloth_zoo': None, 'trl': '==0.22.2',
    'transformers': '==5.2.0', 'tokenizers': '>=0.22.0,<=0.23.0',
    'torchao': '>=0.16.0',
}

missing = []
for pkg, spec in PINS.items():
    try:
        v = md.version(pkg)
    except md.PackageNotFoundError:
        missing.append(pkg + (spec or '')); continue
    if spec and not SpecifierSet(spec).contains(v):
        missing.append(f'{pkg}{spec} (ada {v})')

if not missing:
    print('Dependency lengkap - skip install.')
else:
    print('Install diperlukan:', ', '.join(missing))
    if shutil.which('uv') is None:
        !pip install --upgrade -qqq uv
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = 'numpy'; _pil = 'pillow'
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
    !uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
    !uv pip install transformers==5.2.0
    !uv pip install --no-deps --upgrade "torchao>=0.16.0"
    !unsloth install-kernels
    print('install selesai - jika import unsloth gagal: Runtime -> Restart session')

for pkg in ('torch', 'unsloth', 'trl', 'transformers', 'tokenizers'):
    try: print(pkg, md.version(pkg))
    except md.PackageNotFoundError: print(pkg, 'TIADA')

In [ ]:
# TRAIN: shard demi shard, checkpoint ke Drive, auto-resume.
# Larian semula = sambung; semua shard siap = terus eksport merged + GGUF.
BASE = globals().get('BASE', 'Qwen/Qwen3.5-0.8B')
OUT_DIR = globals().get('OUT_DIR', '/content/drive/MyDrive/DriveXcoder/xcoder/v2')
ARGS = globals().get('ARGS', '')
STREAM = globals().get('STREAM', False)
WORK = globals().get('WORK', '/content' if STREAM else '/content/train-my-ai')
SCRIPT = globals().get('SCRIPT', '/content/train_shreds.py' if STREAM else 'finetune/train_shreds.py')
DELETE = '--delete-after' if STREAM else ''
%cd {WORK}
!python {SCRIPT} --model {BASE} --out {OUT_DIR} {ARGS} {DELETE}

In [ ]:
# Semak hasil
OUT_DIR = globals().get('OUT_DIR', '/content/drive/MyDrive/DriveXcoder/xcoder/v2')
!ls -lh {OUT_DIR}
!ls -lh {OUT_DIR}/gguf

## Selepas ini (di PC)
1. Muat turun `OUT_DIR/gguf/*.gguf` dari Drive.
2. Daftar: `/model add <fail> --name xcoder-gen2 --domains general`
3. Uji identiti: "Siapa anda?" / "Are you Qwen?" -> "Saya XCoder..." tanpa sebut Qwen/Alibaba.

Nota: `OUT_DIR/merged` ialah safetensors penuh - muat naik ke HF jika mahu teruskan
training dari gen2 nanti (`--model <repo-merged>`).